In [ ]:
import functools

import jax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

import numpy as np

from sfp.utils import benchmark, numerics

In [ ]:
m, k, n = 16384, 16384, 8192

k1, k2 = jax.random.split(jax.random.key(0), 2)
lhs = jax.random.normal(k1, (m, k), dtype=jnp.bfloat16)
rhs = jax.random.normal(k2, (k, n), dtype=jnp.bfloat16)

In [ ]:
num_devices = jax.device_count()
mesh = jax.make_mesh((2, 2), ("x_mesh", "y_mesh"))
lhs_sharding = NamedSharding(mesh, P('x_mesh', 'y_mesh'))
rhs_sharding = NamedSharding(mesh, P('x_mesh', None))

inputs = jax.device_put(lhs, lhs_sharding)
weights = jax.device_put(rhs, rhs_sharding)

In [ ]:
def jax_matmul(x: jax.Array, y: jax.Array) -> jax.Array:
    return jnp.matmul(x, y)

ref = jax_matmul(lhs, rhs)

In [ ]:
ref

In [ ]:
jax_matmul_compiled = jax.jit(jax_matmul)
jmc = jax_matmul_compiled(lhs, rhs)
jmc.block_until_ready()

with jax.profiler.trace('./traces/naive_matmul'):
    result = jax_matmul_compiled(lhs, rhs)
    result.block_until_ready()

In [ ]:
"""
func_compiled = func.lower(x, y).compile({'xla_enable_transpose_trace': True})
res = func_compiled(x, y)
res.block_until_ready()

with jax.profiler.trace('./traces/func'):
    res = func_compiled(lhs, rhs)
    result.block_until_ready()
"""